# Intro & Setup

## What's covered

- What Scala is and the niche it occupies between Java, Python, and Kotlin
- Why Scala 3 — what changed from Scala 2 and what to expect when you bump into older code
- The JVM contract: `scalac`, bytecode, JIT, garbage collector
- Java interop — every JAR on Maven Central is a usable library
- When NOT to use Scala
- Installing the whole toolchain in one shot with Coursier
- Three on-ramps: REPL for one-liners, `scala-cli` for single files, `sbt` for real projects
- `@main` entry points and the `Unit` type
- Indentation vs brace syntax — pick one per file
- Comments and ScalaDoc
- How Spark uses Scala — the reading-other-people's-code reality you'll meet inside `apache-spark/`

## What Scala is

Scala is a **statically typed, hybrid functional and object-oriented** language that targets the **Java Virtual Machine** (JVM).

Martin Odersky designed it at EPFL in 2004 with one goal: keep the engineering pragmatism of Java — the libraries, the JVM, the tooling — and fuse it with the expressive power of ML-family functional languages. The name is a contraction of *scalable language* — the same syntax should feel right for a five-line script and for a million-line distributed system.

Four traits that define Scala in one breath:

- **Statically typed, but inference-heavy.** The compiler catches type errors at build time. You rarely write the types yourself — the compiler infers most of them.
- **Functions are values.** You pass them as arguments, return them from other functions, store them in collections, exactly like integers or strings.
- **Immutable by default.** `val` (a single-assignment binding) is preferred over `var` (a mutable variable). Standard-library collections are immutable unless you explicitly import the mutable ones.
- **Runs on the JVM.** Every Java library on Maven Central is reachable from Scala with zero ceremony — same classpath, same `.class` files, same bytecode.

## Why Scala — and where it sits next to its neighbours

Three reasons people reach for Scala in 2026:

| Reason | What it unlocks |
|---|---|
| **Spark and big-data engineering** | Apache Spark itself is written in Scala. The native Spark API, the optimizer internals, the tuning knobs — all expressed in Scala first. Python bindings come second. |
| **Type-safe domain modeling** | Algebraic data types, exhaustive pattern matching, generics with variance, type classes — the type system encodes invariants the compiler will enforce. Whole classes of bugs become unrepresentable. |
| **JVM ecosystem with functional ergonomics** | Every Java library, every Kafka client, every Hadoop tool — but you write the glue with `map`, `flatMap`, and pattern matching instead of `for (int i = 0; i < n; i++)`. |

A quick comparison against the three neighbouring languages this user is likely to be coming from:

| Versus | Scala's pitch |
|---|---|
| **Java** | Same JVM, far less ceremony. No `public static void main`, no getters and setters, no `new` keyword required, type inference for most locals, immutability as the default. Trades some compile-time speed and IDE refactoring depth for expressiveness. |
| **Python** | Static types catch errors before run, and the same code runs an order of magnitude faster on the JVM than CPython on a hot path. Trades Python's REPL-first ergonomics and gentle learning curve for a stricter compiler and a slower startup. |
| **Kotlin** | Both target the JVM and both fix Java's main pain points. Kotlin is the more conservative redesign — easy adoption from Java, paired tightly with Android and JetBrains tooling. Scala is the more ambitious one — a deeper type system, true pattern matching, type classes via `given`. Scala wins when domain modeling is the hard part; Kotlin wins when you want a smoother on-ramp from Java.

For this learning track, the centre of gravity is the **Spark connection**. Every later notebook will, where natural, point at the Scala feature being used inside Spark's own codebase.

## Scala 3, not Scala 2

This track teaches **Scala 3** — the version line that landed in 2021, codename *Dotty* during its long incubation.

Scala 3 is a deep redesign of the language: same JVM target, same flavour, but a cleaner type system, fewer footguns, and a much friendlier surface syntax. The changes you will feel immediately:

- **Significant indentation.** Code blocks can be expressed with indentation the way Python does, instead of curly braces. Braces still work; you pick one style per file.
- **`given` and `using` replace `implicit`.** Scala 2's most powerful and most confusing feature was rebuilt with clearer keywords. You will meet these in notebook 08.
- **`enum` is a first-class keyword.** Algebraic data types — `sealed trait` plus a handful of `case class` extending it — get a one-keyword form. Notebook 05 covers this.
- **`extension` methods.** Add methods to types you don't own without the old `implicit class` trick. Also in notebook 08.
- **Top-level definitions.** You can declare a `def` or `val` at the top of a file, no enclosing `object` required.

If you stumble onto a Stack Overflow answer using `implicit val foo` or `object Main extends App`, that is Scala 2 vintage. The shape of the idea usually carries over; only the keyword differs.

One practical note: most of the JVM library ecosystem still publishes for **Scala 2.13** as well as Scala 3. Scala 3 is binary-compatible with Scala 2.13 libraries at the bytecode level, so you can depend on a Scala 2.13 library from a Scala 3 project. The reverse — depending on a Scala 3 library from a Scala 2 project — does not work. Spark is the big exception you will care about: as of 2026 Spark still targets Scala 2.13, so when this track bridges to Spark you will be reading Scala 2 source code.

## The JVM context

When you write Scala, the compiler `scalac` turns your source into **JVM bytecode** — the same instruction set Java compiles to. The JVM then loads that bytecode and executes it.

```text
   hello.scala
       |
       v
   [ scalac ]                <- Scala 3 compiler
       |
       v
   hello.class               <- JVM bytecode (.class file)
       |
       v
   [ JVM at runtime ]        <- class loader, garbage collector,
       |                        JIT compiler for hot paths
       v
     output
```

Three consequences worth absorbing now.

**JVM startup is not free.** The JVM itself takes a moment to boot — typically a few hundred milliseconds before your `main` runs. For tiny scripts that feels slow; for long-running services it disappears into the noise. The `scala-cli` runner mitigates this for one-off scripts by reusing a warm JVM where it can.

**You inherit the JVM's strengths.** A mature garbage collector frees you from manual memory management. The JIT compiler watches hot code paths and recompiles them down to optimized machine code at runtime — so a method called a million times can end up faster in JVM Scala than in ahead-of-time-compiled C, in some shapes. The profiling and observability ecosystem (JFR, async-profiler, JMC, VisualVM) all work on Scala bytecode unchanged.

**You inherit the JVM's footprint.** A Scala process holds the JVM in memory — typically tens of megabytes of resident set even for trivial programs. For a server that is invisible; for a CLI tool you ship to laptops, it's a real consideration. (GraalVM Native Image is the escape hatch when you really need AOT-compiled standalone binaries; it works with Scala but adds build complexity.)

## Java interop — every JAR is yours

Because Scala compiles to JVM bytecode that looks indistinguishable from Java's, **any Java library is automatically a Scala library**. There is no foreign function interface, no binding layer, no wrappers needed.

Practically this means:

- The entire Java standard library is in scope. `java.time.LocalDate`, `java.util.UUID`, `java.nio.file.Path` — all callable from Scala with the same method signatures.
- Every artifact on Maven Central — the JVM's package registry — is one line in `build.sbt` away from being a dependency.
- A Scala class can extend a Java class; a Java class can extend a Scala class. Same `extends`, same vtable, same single-inheritance rules.

The friction points, in order of how often you'll trip over them:

- **`null`.** Java APIs return `null` to signal absence; Scala APIs use `Option`. Wrap Java returns in `Option(...)` at the boundary to stop nulls leaking into your Scala code. Covered in notebook 06.
- **Checked exceptions.** Java has them; Scala does not. You catch them in Scala using the same `try`/`catch`, but the compiler will not force you to. Notebook 06 again.
- **Collections conversions.** A `java.util.List` is not a `scala.collection.List`. Import `scala.jdk.CollectionConverters.*` and call `.asScala` / `.asJava` to convert at the seam.

You will see all three of these in the Spark code you read later — Spark's Java-API layer is the canonical example of working across the boundary.

## When NOT to use Scala

Three honest cases where reaching for Scala is the wrong call.

- **You need fast cold-start CLIs or AWS Lambdas.** JVM startup latency dominates short-lived processes. Go, Rust, or a Python script will give your users a faster command-line tool with less effort. (GraalVM Native Image is an escape hatch but adds real build complexity.)
- **The team's centre of gravity is Python or Java.** Scala has a real learning curve — implicits, variance, the type system, two competing FP ecosystems (Typelevel vs ZIO). If nobody on the team will mentor newcomers through it, you will pay maintenance cost for years.
- **You only need a one-off scripted ETL.** A Python script with `pandas` or `duckdb` is faster to write, easier to share, and runs everywhere. Reach for Scala when something downstream — Spark, type safety on a big domain model, JVM library access — actually demands it.

The honest rule: Scala pays for itself when you'd otherwise be writing a lot of Java, or when type-safe domain modeling is the hard part. For everything else, the simpler tool usually wins.

## Installing the toolchain — Coursier

The recommended installer is **Coursier** — a single tool that fetches a compatible JDK, the Scala 3 compiler, the REPL, `scala-cli`, `sbt`, and `scalafmt` in one shot.

On macOS the path is two commands. Run them in your terminal, not in this notebook — they touch your shell config.

In [ ]:
%%bash
brew install coursier
cs setup

`cs setup` does three things:

- Installs a JDK if one isn't already on your `PATH` (defaults to a recent Temurin build).
- Drops launchers for `scala`, `scala-cli`, `sbt`, `scalafmt`, and friends into `~/Library/Application Support/Coursier/bin` (or `~/.local/share/coursier/bin` on Linux).
- Adds that directory to your shell `PATH` by editing `.zshrc` or `.bashrc`.

Open a fresh terminal after it finishes so the updated `PATH` takes effect. Then verify each piece responds.

In [ ]:
%%bash
scala --version       # expect: Scala code runner version 3.x.x
scala-cli --version   # quick single-file runner
sbt --version         # multi-file project build tool
java --version        # the JDK Coursier picked

**If `java --version` reports a version below 17:** Scala 3 supports Java 8 and up, but Spark 3.5+ and most of the modern ecosystem expect **Java 17 or 21**. Install one with `brew install --cask temurin@21` and re-run `cs setup`. The track will assume Java 17+ from here on.

**If a `cs` command errors with "command not found":** the launcher directory isn't on your `PATH`. Open a fresh terminal first; if that doesn't help, source your shell config explicitly with `source ~/.zshrc` (zsh) or `source ~/.bashrc` (bash).

## Three on-ramps — pick by size

Scala gives you three ways to run code. Pick the one that matches the size of what you're doing.

```text
   +---------------+   +-----------------+   +---------------------+
   |     REPL      |   |    scala-cli    |   |         sbt         |
   |   one-liner   |   |    one file     |   |    real project     |
   +---------------+   +-----------------+   +---------------------+
    live eval         single .scala       many files, deps,
    no file at all    deps via comment    tests, build config,
                      at top of file      tasks
```

- **REPL** — for *what does this expression actually do?* The cheapest feedback loop in the language. Open it, type, see the result and the inferred type on the same line.
- **`scala-cli`** — for a single file you want to keep around. Add dependencies via a magic comment at the top of the file. The tool was donated by Virtus Lab to the Scala Center; as of 2026 it is the official Scala runner.
- **`sbt`** — for anything that has more than one file or that you want to build, package, and test seriously. This is what Spark itself uses, and what notebook 08 will lean on for the type-class examples.

## On-ramp 1 — REPL

Open the REPL by running `scala` with no arguments. You get a `scala>` prompt that evaluates one expression at a time, echoes the result back with its inferred type, and binds it to an automatic name (`res0`, `res1`, ...) so you can reuse it.

The transcript below is what you would type and what the REPL prints back. Lines starting with `scala>` are your input; lines without that prefix are the REPL's response.

In [ ]:
scala> println("hello, scala 3")
hello, scala 3

scala> val name = "ganesh"
val name: String = ganesh

scala> s"hello, $name"
val res0: String = hello, ganesh

scala> res0.length
val res1: Int = 13

scala> :type res0
String

scala> :quit

Four details to register from that transcript.

**The REPL infers and echoes the type.** `val name = "ganesh"` is one line of input, but the REPL shows you that the inferred type is `String`. This is the fastest way to ask the compiler "what type would this be?" — type the expression, read the answer.

**`s"..."` is a string interpolator.** The `s` prefix lets `$name` and `${...}` be spliced into the string. The compiler turns it into a `StringBuilder` chain at compile time, not at runtime — so there's no parsing cost. Other built-in interpolators include `f"..."` for `printf`-style formatting and `raw"..."` for disabling backslash escapes.

**`res0` is real.** Every expression that doesn't get a `val` name is bound to the next `res<N>` slot. You can reuse it on later lines, as the `res0.length` call shows. Handy for quick exploration.

**Colon commands.** Lines starting with `:` are REPL commands, not Scala. `:help` lists them all. `:type expr` shows the inferred type without evaluating. `:quit` exits.

## On-ramp 2 — `scala-cli`

For anything longer than two lines, put it in a `.scala` file and run it with `scala-cli`. Create a file called `hello.scala` with this content:

In [ ]:
@main def hello(): Unit =
  println("hello from scala-cli")

Run it from the same directory:

In [ ]:
%%bash
scala-cli run hello.scala
# hello from scala-cli

Two new shapes appeared in that tiny file.

**`@main def hello()` — the entry point annotation.** Any top-level method tagged `@main` is callable as a program. The method name becomes the command name (`scala-cli run hello.scala` looks for an `@main` method to call). No more `object Main extends App` boilerplate that Scala 2 needed; no `public static void main(String[] args)` ceremony that Java needs.

**`: Unit =` — the return type.** `Unit` is Scala's equivalent of `void`. The difference: `Unit` is a real type with exactly one value, written `()`. That sounds pedantic, but it matters when functions are values — a function that "returns nothing" still has a real return type you can write down. We'll lean on this in notebook 03.

The `@main` method can also take parameters, and `scala-cli` will parse the command-line arguments to match them:

In [ ]:
@main def greet(name: String, times: Int): Unit =
  for _ <- 1 to times do
    println(s"hello, $name")

// Run as:
//   scala-cli run greet.scala -- ganesh 3
// The -- separates scala-cli's own flags from arguments to your program.

**Adding dependencies in a single file.** `scala-cli` reads magic comments starting with `//>` at the top of the file. The most common ones:

```scala
//> using scala 3.4.0
//> using dep com.lihaoyi::upickle:3.1.4
//> using dep com.lihaoyi::os-lib:0.10.0

import upickle.default.*
import os.*

@main def main(): Unit =
  val files = os.list(os.pwd).map(_.last)
  println(write(files))
```

`scala-cli` resolves the dependencies, compiles, and runs — no `build.sbt`, no project directory. For exploring a library or writing a small tool, this is the fastest path.

## On-ramp 3 — `sbt`

For a real project, **sbt** (the *simple build tool*) gives you a directory layout, dependency management, incremental compilation, and a test runner. You will meet it properly when this track bridges to Spark; for now, the scaffold and the layout are enough.

Generate a fresh project from the official Scala 3 template:

In [ ]:
%%bash
sbt new scala/scala3.g8

sbt downloads the template, asks for a project name, and creates a directory with the standard layout:

```text
   myproject/
   +-- build.sbt              <- project config: name, version, scala version, deps
   +-- project/
   |    +-- build.properties  <- which sbt version to use
   +-- src/
   |    +-- main/scala/       <- production code
   |    +-- test/scala/       <- tests
   +-- target/                <- compiled output and cache (gitignored)
```

A minimal `build.sbt` looks like this:

```scala
ThisBuild / scalaVersion := "3.4.0"
ThisBuild / version      := "0.1.0-SNAPSHOT"

lazy val root = (project in file("."))
  .settings(
    name := "myproject",
    libraryDependencies ++= Seq(
      "org.scalameta" %% "munit" % "0.7.29" % Test
    )
  )
```

The shapes worth noting at this stage:

- `:=` is the sbt operator for "set this setting to this value." `++=` is "append to this collection setting."
- `%%` is Scala-aware dependency notation — it appends the Scala binary version to the artifact name automatically, so the same line works whether you're on Scala 3.4 or Scala 3.5.
- `% Test` scopes a dependency to the test configuration only.

Run the project from the directory with `sbt run`, or open the sbt shell with just `sbt` and then issue commands interactively (`compile`, `run`, `test`, `~test` for continuous watch-and-retest).

## Indentation vs brace syntax

Scala 3 supports **two equivalent surface syntaxes** for code blocks. The same method, both ways:

In [ ]:
// Indentation style — Scala 3 idiomatic
def greet(name: String): String =
  val greeting = s"hello, $name"
  greeting.toUpperCase

// Brace style — still valid, common in older code and in Spark
def greet(name: String): String = {
  val greeting = s"hello, $name"
  greeting.toUpperCase
}

Both parse to the same syntax tree. Pick whichever reads better for the file you're editing — and ideally stay consistent within a file.

This track defaults to **indentation style** for two reasons:

- It matches the Scala 3 idiomatic direction the language designers nudge you toward.
- It removes visual noise from short methods — and most methods you write should be short.

You will see **brace style** when you read Spark's source code, because Spark still targets Scala 2.13. Don't let the brace style throw you — it's the same language.

One subtlety: in indentation style, the `end` marker can optionally close a long block, mainly for clarity on long methods or classes. It's never required:

```scala
def longMethod(): Unit =
  // ... 40 lines ...
  println("done")
end longMethod
```

Use `end` sparingly. If you're tempted to add it for readability, the method is probably too long and should be split first.

## Comments and ScalaDoc

Same conventions as Java and C, with one Scala addition (`ScalaDoc`):

In [ ]:
// single-line comment

/*
   multi-line block comment
*/

/** ScalaDoc comment — used to document public APIs.
  *
  * Tools like `scaladoc` turn these into HTML reference docs.
  *
  * @param name the name to greet
  * @return     a greeting string
  */
def greet(name: String): String = s"hello, $name"

ScalaDoc tags worth knowing: `@param`, `@return`, `@throws`, `@see`, `@example`. The official Scala 3 standard library docs at `scala-lang.org/api` are themselves ScalaDoc output — that's the shape you're producing.

## How Spark uses Scala — what to expect when you read its source

Apache Spark itself is written in Scala. When you eventually crack open the Spark source — and this track's `apache-spark/` repo will give you many reasons to — there are five shapes you'll see over and over. None of them will be a surprise by the end of this track:

| You'll see | What it is | Covered in |
|---|---|---|
| `case class Row(values: Array[Any])` | A case class — immutable data with auto-generated `equals`, `hashCode`, `toString`, pattern matching | Notebook 05 |
| `df.filter(_.amount > 100)` | A lambda passed where a `Function1` is expected — functions as values | Notebook 03 |
| `sealed trait JoinType; case object Inner extends JoinType` | An ADT — closed set of subtypes the compiler can check exhaustively | Notebooks 05 + 06 |
| `implicit val enc: Encoder[T] = ...` | A type-class instance (Scala 2 syntax — Scala 3's `given` is the same idea) | Notebook 08 |
| `Future { expensive() }.map(_ + 1)` | Asynchronous computation chained with combinators | Notebook 09 |

Spark also still uses **brace syntax** and **Scala 2.13** throughout. The constructs are identical; only the surface syntax differs. You will not be reading unfamiliar territory — you will be reading the same language with curly braces and an older keyword for `given`.

## What's next

You now have Scala 3 installed, you can poke at it in the REPL, you can run a one-file script with `scala-cli`, and you know what an `sbt` project looks like.

Notebook 02 starts the language proper: `val` versus `var`, the type system you mostly cannot see, and the fact that `if`, `while`, and `for` are **expressions** that return values — a small mental shift with large downstream consequences for how Scala code reads.